# Living Canvas: real-model test v2

**What v1 showed:** the opening frame looked like Monet, but chaining img2img (each frame drawn from the previous one) collapsed into yellow, blue and white speckle by visitor 2 or 3. That happened in every variant, including plain text prompts, so the loop was the cause, not the embedding math. Visitors 4–10 looked identical, and a strength-scaling bug pinned almost everyone at about 0.33.

**What v2 changes:**
1. **History lives in the crowd vector, not the pixels.** Every frame is drawn fresh from the same opening painting and seed. Only the conditioning changes, so nothing compounds.
2. **Visitors are directions, not prompts.** Each visitor's direction is `d = E(style + their reading + tone + movement) − E(style)`. The prompts share the same style prefix, so token positions line up and the subtraction is meaningful.
3. **Crowd update:** `D ← D + α(d − D)`, where `α` is bigger when your direction differs from the crowd's. Conditioning = `E(style) + gain · D`.
4. **Strength sweep** (0.45, 0.6, 0.75) to find where changes become visible but the chapel survives.
5. **A morph GIF** that blends the crowd vector between visitors: the booth's living painting.
6. **Part B (optional): structure lock.** SDXL-Turbo plus a Canny ControlNet holds the chapel's outlines while each frame is generated fresh from noise, so light, weather and season can change completely.

**Pass criteria:** (1) all frames are clearly the same chapel and pond, (2) you can tell which visitor caused each change, (3) it looks like a painting, (4) under about 3 s per frame.

**Setup:** Colab → *Runtime → Change runtime type → T4 GPU*, on a personal or Duke Google account. Part A takes about 5 minutes. Part B downloads about 9 GB more and takes 10–15 minutes.

In [ ]:
!pip -q install "diffusers==0.31.0" "transformers==4.46.3" "accelerate==1.1.1"

In [ ]:
import torch, time, os
assert torch.cuda.is_available(), "No GPU found. In Colab: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))
os.makedirs("out", exist_ok=True)

# Part A: anchor decode with SD-Turbo

In [ ]:
from diffusers import AutoPipelineForText2Image, AutoPipelineForImage2Image
t0 = time.time()
t2i = AutoPipelineForText2Image.from_pretrained("stabilityai/sd-turbo", torch_dtype=torch.float16, variant="fp16").to("cuda")
i2i = AutoPipelineForImage2Image.from_pipe(t2i)
t2i.set_progress_bar_config(disable=True); i2i.set_progress_bar_config(disable=True)
W, H = 768, 432
print(f"loaded in {time.time() - t0:.0f}s")

In [ ]:
STYLE = ("an impressionist oil painting by Claude Monet, a gothic chapel tower above a garden pond, "
         "reflections on the water, thick visible brushstrokes")
TONES = {
    "whisper": "hushed, soft, dim, quiet",
    "calm": "calm, still, gentle",
    "warm": "warm golden glow, tender",
    "excited": "bright, vivid, lively",
    "shouting": "loud, turbulent, dramatic, stormy",
}
MOVES = {
    "float": "drifting, airy, floating mist",
    "glide": "smooth, serene, gliding",
    "dab": "light dabs of color, sparkling",
    "flick": "flickering, playful",
    "press": "heavy, pressing, dense",
    "wring": "twisting, wrung, tense",
    "punch": "forceful, explosive",
    "slash": "sweeping, slashing, windswept",
}
# (what they said, visual reading, tone, movement): the 10 example visitors from the web demo
VISITORS = [
    ("Finally submitted my thesis. I could sleep for a week.", "dusk settles, lilies open on still water", "calm", "float"),
    ("Everyone I love is in this room right now.", "golden light, the banks fill with people", "warm", "glide"),
    ("I am so stressed about the job market.", "a storm gathers over cold water", "shouting", "wring"),
    ("Missing my grandma tonight.", "night mist, an empty bank", "whisper", "press"),
    ("WE WON THE GAME!!", "a bright roar of color, crowds at the water", "shouting", "punch"),
    ("First snow of the year on the walk here.", "winter light on the tower, snow on the trees", "calm", "float"),
    ("Honestly no idea what I am doing with my life.", "thick fog swallows the far bank", "whisper", "wring"),
    ("Spring break in nine days!", "bright morning, the garden in full bloom", "excited", "flick"),
    ("I want to build something that matters.", "the air clears, the tower stands lit", "warm", "press"),
    ("Thinking about the people who are not here.", "lamplight in the windows, a quiet bank", "whisper", "glide"),
]
def visitor_prompt(text, tone, move):
    return f"{STYLE}, {text}, {TONES[tone]}, {MOVES[move]}"

@torch.no_grad()
def E(text):
    """CLIP text embedding: the conditioning the model normally gets from a prompt. Shape (1, 77, 1024)."""
    emb, _ = t2i.encode_prompt(text, device="cuda", num_images_per_prompt=1, do_classifier_free_guidance=False)
    return emb

def cos_dist(a, b):
    a, b = a.flatten().float(), b.flatten().float()
    if b.norm() < 1e-6:
        return 1.0
    return 1 - torch.nn.functional.cosine_similarity(a, b, dim=0).item()

def crowd_alphas(deltas):
    """alpha per visitor: 0.35 when you point the same way as the crowd, up to 0.7 when you point somewhere new."""
    D, alphas = torch.zeros_like(deltas[0]), []
    for d in deltas:
        a = 0.35 + 0.35 * min(cos_dist(d, D), 1.0)
        D = D + a * (d - D); alphas.append(round(a, 2))
    return alphas

def crowd_path(deltas, alphas):
    """The crowd direction D after each visitor."""
    D, out = torch.zeros_like(deltas[0]), []
    for d, a in zip(deltas, alphas):
        D = D + a * (d - D); out.append(D.clone())
    return out

GAIN = 1.2   # >1 exaggerates the crowd's pull so changes read on a big screen
def gen():
    return torch.Generator("cuda").manual_seed(7)   # fixed seed is safe now: nothing is chained

S_EMB = E(STYLE)
deltas = [E(visitor_prompt(r, t, m)) - S_EMB for _, r, t, m in VISITORS]
alphas = crowd_alphas(deltas)
path = crowd_path(deltas, alphas)
print("alpha per visitor:", alphas)

In [ ]:
from PIL import Image, ImageDraw

def decode(cond, strength):
    """Draw a frame fresh from the opening painting with this conditioning. strength = how far it may move from the opening."""
    return i2i(prompt_embeds=cond, image=base, strength=strength, num_inference_steps=4,
               guidance_scale=0.0, generator=gen()).images[0]

def grid(frames, labels, cols=4):
    w, h = frames[0].size; tw, th = w // 2, h // 2
    rows = (len(frames) + cols - 1) // cols
    G = Image.new("RGB", (cols * tw, rows * (th + 24)), "white"); d = ImageDraw.Draw(G)
    for i, (f, lab) in enumerate(zip(frames, labels)):
        x, y = (i % cols) * tw, (i // cols) * (th + 24)
        G.paste(f.resize((tw, th)), (x, y)); d.text((x + 4, y + th + 5), lab[:62], fill="black")
    return G

def row(frames):
    w, h = frames[0].size; R = Image.new("RGB", (w * len(frames) // 2, h // 2))
    for i, f in enumerate(frames): R.paste(f.resize((w // 2, h // 2)), (i * w // 2, 0))
    return R

## Smoke test: opening frame, then two very different visitors at each strength

In [ ]:
t0 = time.time()
base = t2i(prompt_embeds=S_EMB, num_inference_steps=1, guidance_scale=0.0, width=W, height=H, generator=gen()).images[0]
print(f"opening frame: {time.time() - t0:.2f}s"); display(base)

for i in (1, 3):   # "Everyone I love..." (warm crowd) and "Missing my grandma" (night mist)
    one = crowd_path([deltas[i]], [0.7])[0]
    t0 = time.time(); frames = [decode(S_EMB + GAIN * one, s) for s in (0.45, 0.6, 0.75)]
    print(f"{VISITORS[i][0]!r}: strengths 0.45 / 0.6 / 0.75, {(time.time() - t0) / 3:.2f}s per frame")
    display(row(frames))

## Full sweep: 10 visitors at 3 strengths

Each frame shows the painting after that visitor joins the crowd. Look for the strength where frame-to-frame changes are clearly visible but it is still the same chapel.

In [ ]:
for s in (0.45, 0.6, 0.75):
    times, frames = [], [base]
    for D in path:
        t = time.time(); frames.append(decode(S_EMB + GAIN * D, s)); times.append(time.time() - t)
    labels = ["opening"] + [f"{i + 1}. {VISITORS[i][1][:36]} a={alphas[i]}" for i in range(len(path))]
    G = grid(frames, labels); G.save(f"out/anchor_s{int(s * 100)}_grid.jpg", quality=90)
    frames[0].save(f"out/anchor_s{int(s * 100)}.gif", save_all=True, append_images=frames[1:], duration=900, loop=0)
    print(f"\n=== strength {s} === median {sorted(times)[len(times) // 2]:.2f}s per frame"); display(G)

## Optional: raw words instead of the visual reading

v1 showed little difference, but here is the same comparison under the new setup. Set `S_RAW` to your best strength from the sweep.

In [ ]:
S_RAW = 0.6
raw_deltas = [E(visitor_prompt(w, t, m)) - S_EMB for w, _, t, m in VISITORS]
raw_path = crowd_path(raw_deltas, crowd_alphas(raw_deltas))
frames = [base] + [decode(S_EMB + GAIN * D, S_RAW) for D in raw_path]
G = grid(frames, ["opening"] + [f"{i + 1}. {VISITORS[i][0][:40]}" for i in range(len(raw_path))])
G.save(f"out/raw_words_s{int(S_RAW * 100)}_grid.jpg", quality=90); display(G)

## Morph: the living painting

Blends the crowd vector smoothly from each visitor to the next (6 in-between frames each). This is what the booth screen would show between people. Set `S_BEST` from the sweep.

In [ ]:
S_BEST, TWEENS = 0.6, 6
morph, prev, t0 = [], torch.zeros_like(path[0]), time.time()
for D in path:
    for k in range(1, TWEENS + 1):
        morph.append(decode(S_EMB + GAIN * (prev + (D - prev) * k / TWEENS), S_BEST))
    prev = D
morph[0].save(f"out/morph_s{int(S_BEST * 100)}.gif", save_all=True, append_images=morph[1:], duration=120, loop=0)
print(f"{len(morph)} morph frames, {(time.time() - t0) / len(morph):.2f}s each -> out/morph_s{int(S_BEST * 100)}.gif")
display(row(morph[::TWEENS][:6]))

# Part B (optional): structure lock with SDXL-Turbo + ControlNet

Holds the opening painting's outlines (Canny edges) while every frame is generated **fresh from noise**, so night, snow and storms are reachable. Frees Part A's model first. Downloads about 9 GB.

In [ ]:
import gc, numpy as np, cv2
del t2i, i2i; gc.collect(); torch.cuda.empty_cache()
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel, AutoencoderKL
t0 = time.time()
try:
    cn = ControlNetModel.from_pretrained("diffusers/controlnet-canny-sdxl-1.0", torch_dtype=torch.float16, variant="fp16")
except Exception:
    cn = ControlNetModel.from_pretrained("diffusers/controlnet-canny-sdxl-1.0", torch_dtype=torch.float16)
vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
xl = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/sdxl-turbo", controlnet=cn, vae=vae, torch_dtype=torch.float16, variant="fp16").to("cuda")
xl.set_progress_bar_config(disable=True)
print(f"loaded in {time.time() - t0:.0f}s")

edges = cv2.Canny(np.array(base), 80, 160)
edge_img = Image.fromarray(np.stack([edges] * 3, -1)); display(edge_img.resize((W // 2, H // 2)))

In [ ]:
@torch.no_grad()
def E_xl(text):
    pe, _, pp, _ = xl.encode_prompt(prompt=text, device="cuda", num_images_per_prompt=1, do_classifier_free_guidance=False)
    return pe, pp

CN_SCALE = 0.55   # how tightly to hold the outlines: lower = freer, higher = stiffer
def decode_xl(pe, pp):
    return xl(prompt_embeds=pe, pooled_prompt_embeds=pp, image=edge_img, controlnet_conditioning_scale=CN_SCALE,
              num_inference_steps=4, guidance_scale=0.0, width=W, height=H, generator=gen()).images[0]

S_PE, S_PP = E_xl(STYLE)
enc = [E_xl(visitor_prompt(r, t, m)) for _, r, t, m in VISITORS]
d_pe, d_pp = [pe - S_PE for pe, _ in enc], [pp - S_PP for _, pp in enc]
xl_alphas = crowd_alphas(d_pe)
path_pe, path_pp = crowd_path(d_pe, xl_alphas), crowd_path(d_pp, xl_alphas)

t0 = time.time(); opening = decode_xl(S_PE, S_PP); print(f"opening: {time.time() - t0:.2f}s")
times, frames = [], [opening]
for Dp, Dq in zip(path_pe, path_pp):
    t = time.time(); frames.append(decode_xl(S_PE + GAIN * Dp, S_PP + GAIN * Dq)); times.append(time.time() - t)
labels = ["opening"] + [f"{i + 1}. {VISITORS[i][1][:36]} a={xl_alphas[i]}" for i in range(len(path_pe))]
G = grid(frames, labels); G.save("out/structure_lock_grid.jpg", quality=90)
frames[0].save("out/structure_lock.gif", save_all=True, append_images=frames[1:], duration=900, loop=0)
print(f"median {sorted(times)[len(times) // 2]:.2f}s per frame"); display(G)

## Score it and download

For each run (the 3 strengths, raw words, structure lock), note pass or fail on the four criteria and one line on what surprised you.

In [ ]:
!zip -qr living_canvas_v2_results.zip out
try:
    from google.colab import files
    files.download("living_canvas_v2_results.zip")
except ImportError:
    print("Not in Colab: results are in ./out and living_canvas_v2_results.zip")